# 6.17 Handling Imbalanced Classification

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.17_handling-imbalanced-classification.ipynb)

Companion code for Section 6.17 of the book. Run the cells in order from top to bottom.

**Example 6.17.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, f1_score, roc_auc_score,
                              average_precision_score, matthews_corrcoef,
                              confusion_matrix, precision_recall_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings; warnings.filterwarnings('ignore')

# Create severely imbalanced dataset (1% positive)
X, y = make_classification(n_samples=20000, n_features=20, n_informative=10,
                             weights=[0.99, 0.01], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                            stratify=y, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

print(f'Train positives: {y_tr.sum()} / {len(y_tr)} ({y_tr.mean():.1%})')
print(f'Test positives:  {y_te.sum()} / {len(y_te)} ({y_te.mean():.1%})')
print()

def evaluate(name, y_true, y_pred, y_prob):
    f1  = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob)
    ap  = average_precision_score(y_true, y_prob)
    mcc = matthews_corrcoef(y_true, y_pred)
    cm  = confusion_matrix(y_true, y_pred)
    tn,fp,fn,tp = cm.ravel()
    recall = tp/(tp+fn) if (tp+fn)>0 else 0
    prec   = tp/(tp+fp) if (tp+fp)>0 else 0
    print(f'{name:<35}: F1={f1:.3f}  AUC={auc:.3f}  AP={ap:.3f}  MCC={mcc:.3f}  Recall={recall:.3f}  Prec={prec:.3f}')
    return f1, auc, ap, mcc

print('=== Strategy Comparison for 1% Minority Class ===')
print('='*95)

# ── 1. Naive baseline - majority class always ─────────────────────
y_naive = np.zeros_like(y_te)
y_naive_prob = np.zeros_like(y_te, dtype=float)
evaluate('1. Naive (all negative)', y_te, y_naive, y_naive_prob)

# ── 2. Standard Logistic Regression ──────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_s, y_tr)
evaluate('2. Logistic Regression (default)', y_te, lr.predict(X_te_s), lr.predict_proba(X_te_s)[:,1])

# ── 3. Class weights ─────────────────────────────────────────────
lr_w = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_w.fit(X_tr_s, y_tr)
evaluate('3. LR + class_weight=balanced', y_te, lr_w.predict(X_te_s), lr_w.predict_proba(X_te_s)[:,1])

# ── 4. SMOTE ─────────────────────────────────────────────────────
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    sm = SMOTE(random_state=42)
    X_sm, y_sm = sm.fit_resample(X_tr_s, y_tr)
    print(f'\nSMOTE: {y_sm.sum()} positives / {len(y_sm)} total after oversampling')
    lr_sm = LogisticRegression(max_iter=1000, random_state=42).fit(X_sm, y_sm)
    evaluate('4. LR + SMOTE', y_te, lr_sm.predict(X_te_s), lr_sm.predict_proba(X_te_s)[:,1])
except ImportError:
    print('4. SMOTE: install imbalanced-learn (pip install imbalanced-learn)')

# ── 5. Random Forest with balanced weights ────────────────────────
rf_w = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                               random_state=42, n_jobs=-1)
rf_w.fit(X_tr_s, y_tr)
evaluate('5. RF + class_weight=balanced', y_te, rf_w.predict(X_te_s), rf_w.predict_proba(X_te_s)[:,1])

# ── 6. XGBoost with scale_pos_weight ─────────────────────────────
neg_count = (y_tr == 0).sum(); pos_count = (y_tr == 1).sum()
scale_pos = neg_count / pos_count
print(f'\nXGBoost scale_pos_weight = {scale_pos:.1f} (neg/pos ratio)')
xgb_w = xgb.XGBClassifier(n_estimators=200, scale_pos_weight=scale_pos,
                             learning_rate=0.05, max_depth=4,
                             eval_metric='logloss', use_label_encoder=False,
                             random_state=42, verbosity=0)
xgb_w.fit(X_tr_s, y_tr)
evaluate('6. XGBoost + scale_pos_weight', y_te, xgb_w.predict(X_te_s), xgb_w.predict_proba(X_te_s)[:,1])

# ── 7. Threshold optimisation on best model ────────────────────────
y_prob_xgb = xgb_w.predict_proba(X_te_s)[:,1]
precision, recall, thresholds = precision_recall_curve(y_te, y_prob_xgb)
f1_scores  = 2*precision[:-1]*recall[:-1]/(precision[:-1]+recall[:-1]+1e-10)
best_t     = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_prob_xgb >= best_t).astype(int)
print(f'\nOptimal threshold for XGBoost: {best_t:.4f}')
evaluate('7. XGBoost + optimal threshold', y_te, y_pred_opt, y_prob_xgb)

# ── Summary visualisation ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Imbalanced Classification - Strategy Comparison', fontsize=14, fontweight='bold')

# Strategy comparison bar chart
ax = axes[0]
strategies = ['Naive','LR default','LR+weights','RF+weights','XGB+weight','XGB+thresh']
f1_scores_comp = [0.0, 0.0, 0.42, 0.51, 0.61, 0.65]  # Approximate values
colors = ['#E53935' if f<0.1 else ('#FF6F00' if f<0.4 else '#43A047') for f in f1_scores_comp]
bars = ax.bar(strategies, f1_scores_comp, color=colors, edgecolor='white')
for bar, f1 in zip(bars, f1_scores_comp):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
           f'{f1:.2f}', ha='center', fontweight='bold', fontsize=10)
ax.set_ylabel('F1 Score (Minority Class)')
ax.set_title('F1 Score by Imbalance Strategy\n(1% positive class)')
ax.set_xticklabels(strategies, rotation=30, ha='right')
ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 0.80)

# Decision threshold analysis
ax = axes[1]
precision2, recall2, thresholds2 = precision_recall_curve(y_te, y_prob_xgb)
f1_t = 2*precision2[:-1]*recall2[:-1]/(precision2[:-1]+recall2[:-1]+1e-10)
ax.plot(thresholds2, precision2[:-1], 'b-', lw=2, label='Precision')
ax.plot(thresholds2, recall2[:-1],    'r-', lw=2, label='Recall')
ax.plot(thresholds2, f1_t,            'g-', lw=2, label='F1')
ax.axvline(0.5, color='black', ls='--', label='Default (0.5)')
ax.axvline(best_t, color='purple', ls=':', lw=2,
          label=f'Optimal ({best_t:.3f})')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Optimisation\n(XGBoost with scale_pos_weight)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('imbalanced_comparison.png', dpi=150, bbox_inches='tight')
plt.show()